#### P4-05
# Decision Tree Classification Model
#### M2 - Amri B.S.M
#### IT24102955

## 1. Objective

This notebook develops a Decision Tree classifier for hotel booking cancellation prediction.

The modelling workflow includes:

- applying the existing feature-engineering and shared-preprocessing stages
- one-hot encoding categorical predictors inside the model pipeline
- evaluating an untuned Decision Tree baseline
- tuning `max_depth` and `min_samples_leaf`
- using 5-fold stratified cross-validation
- using ROC-AUC as the tuning metric
- exporting readable decision rules
- saving the final fitted pipeline using joblib

The test dataset is not used anywhere in this notebook.

In [8]:
%cd /content

/content


In [9]:
!git clone https://github.com/MahdhiS/hotel-cancellation.git

fatal: destination path 'hotel-cancellation' already exists and is not an empty directory.


In [10]:
%cd /content/hotel-cancellation

/content/hotel-cancellation


In [11]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import (
    DecisionTreeClassifier,
    export_text
)
from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    GridSearchCV
)

import joblib

ROOT = Path("/content/hotel-cancellation")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.preprocessing import split_classifier_data

pd.set_option("display.max_columns", None)

DATA_DIR = ROOT / "data" / "processed" / "shared_prepared"
MODEL_DIR = ROOT / "models"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Prepared data directory:", DATA_DIR)
print("Model directory:", MODEL_DIR)

Project root: /content/hotel-cancellation
Prepared data directory: /content/hotel-cancellation/data/processed/shared_prepared
Model directory: /content/hotel-cancellation/models


## 2. Prepare Training Data

This notebook begins from the shared-prepared training dataset produced by `07_shared_prep.ipynb`.

Feature engineering and shared preprocessing have already been completed before this modelling stage.

Only the prepared training dataset is loaded. Cross-validation is used for model development and the test set remains untouched.

In [12]:
train_prepared = pd.read_csv(
    DATA_DIR / "hotel_train_prepared.csv"
)

print("Prepared training shape:", train_prepared.shape)
print("Missing values:", train_prepared.isna().sum().sum())

train_prepared.head()

Prepared training shape: (58192, 29)
Missing values: 0


,booking_id,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,deposit_type,agent,company,adr,total_nights,total_guests,has_kids,season,arrival_day_of_week
0,2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,BB,Direct,Direct,0,0,0,A,No Deposit,0.0,0.0,75.0,1,1.0,0,Summer,Wednesday
1,3,Resort Hotel,0,13,2015,July,27,1,0,1,1,0.0,0,BB,Corporate,Corporate,0,0,0,A,No Deposit,Other,0.0,75.0,1,1.0,0,Summer,Wednesday
2,4,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,BB,Online TA,TA/TO,0,0,0,A,No Deposit,240.0,0.0,98.0,2,2.0,0,Summer,Wednesday
3,6,Resort Hotel,0,0,2015,July,27,1,0,2,2,0.0,0,BB,Direct,Direct,0,0,0,C,No Deposit,0.0,0.0,107.0,2,2.0,0,Summer,Wednesday
4,7,Resort Hotel,0,9,2015,July,27,1,0,2,2,0.0,0,FB,Direct,Direct,0,0,0,C,No Deposit,Other,0.0,103.0,2,2.0,0,Summer,Wednesday


In [13]:
X_train, y_train, booking_ids = split_classifier_data(
    train_prepared
)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("\nTarget distribution:")
print(y_train.value_counts())

print("\nTarget proportions:")
print(y_train.value_counts(normalize=True))

X_train shape: (58192, 27)
y_train shape: (58192,)

Target distribution:
is_canceled
0    43451
1    14741
Name: count, dtype: int64

Target proportions:
is_canceled
0    0.746683
1    0.253317
Name: proportion, dtype: float64


In [14]:
assert "booking_id" not in X_train.columns
assert "is_canceled" not in X_train.columns
assert "country" not in X_train.columns

print("Predictor checks passed.")

Predictor checks passed.


In [15]:
numeric_features = X_train.select_dtypes(
    include=["number", "bool"]
).columns.tolist()

categorical_features = [
    col for col in X_train.columns
    if col not in numeric_features
]

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Numeric features: 16
Categorical features: 11


## 3. Decision Tree Pipeline

Categorical variables must be converted into numerical form before they can be processed by the Decision Tree.

One-hot encoding is therefore included inside the model pipeline.

Numerical scaling is not applied because Decision Trees make threshold-based splits and are not sensitive to differences in feature scale.

In [16]:
model_preprocessor = ColumnTransformer([
    (
        "numeric",
        "passthrough",
        numeric_features
    ),
    (
        "categorical",
        OneHotEncoder(
            handle_unknown="ignore"
        ),
        categorical_features
    )
])

In [17]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

## 4. Untuned Decision Tree Baseline

An untuned Decision Tree is first evaluated to establish a model-specific baseline.

Decision Trees can easily grow very deep and overfit the training data, so the untuned result is compared with a regularized tree obtained through hyperparameter tuning.

In [18]:
decision_tree_pipeline = Pipeline([
    (
        "preprocessor",
        model_preprocessor
    ),
    (
        "model",
        DecisionTreeClassifier(
            random_state=42
        )
    )
])

In [19]:
baseline_dt_results = cross_validate(
    decision_tree_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring={
        "roc_auc": "roc_auc",
        "pr_auc": "average_precision"
    },
    n_jobs=-1,
    return_train_score=True
)

In [20]:
baseline_dt_table = pd.DataFrame({
    "Fold": range(1, 6),
    "Train ROC-AUC": baseline_dt_results["train_roc_auc"],
    "Validation ROC-AUC": baseline_dt_results["test_roc_auc"],
    "PR-AUC": baseline_dt_results["test_pr_auc"]
})

baseline_dt_table

,Fold,Train ROC-AUC,Validation ROC-AUC,PR-AUC
0,1,0.999755,0.651128,0.365741
1,2,0.999711,0.652503,0.366568
2,3,0.999711,0.656840,0.370344
3,4,0.999724,0.649893,0.363423
4,5,0.999720,0.660137,0.375121


In [21]:
baseline_dt_roc = baseline_dt_results["test_roc_auc"].mean()
baseline_dt_std = baseline_dt_results["test_roc_auc"].std()

baseline_dt_pr = baseline_dt_results["test_pr_auc"].mean()

baseline_dt_train_roc = (
    baseline_dt_results["train_roc_auc"].mean()
)

print(
    "Mean Train ROC-AUC:",
    round(baseline_dt_train_roc, 4)
)

print(
    "Mean CV ROC-AUC:",
    round(baseline_dt_roc, 4)
)

print(
    "CV ROC-AUC std:",
    round(baseline_dt_std, 4)
)

print(
    "Mean CV PR-AUC:",
    round(baseline_dt_pr, 4)
)

Mean Train ROC-AUC: 0.9997
Mean CV ROC-AUC: 0.6541
CV ROC-AUC std: 0.0038
Mean CV PR-AUC: 0.3682


## 5. Hyperparameter Tuning

The Decision Tree is regularized by tuning two complexity-control parameters:

- `max_depth`: limits how deep the tree can grow
- `min_samples_leaf`: specifies the minimum number of observations required in a terminal leaf

These parameters help prevent the tree from fitting very small or highly specific patterns in the training data.

The configurations are evaluated using the shared 5-fold stratified cross-validation setup with ROC-AUC as the optimization metric.

In [22]:
param_grid = {
    "model__max_depth": [
        3,
        5,
        7,
        10,
        15,
        None
    ],
    "model__min_samples_leaf": [
        1,
        5,
        10,
        25,
        50
    ]
}

In [23]:
grid_search_dt = GridSearchCV(
    estimator=decision_tree_pipeline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=True
)

grid_search_dt.fit(
    X_train,
    y_train
)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('numeric',
                                                                         'passthrough',
                                                                         ['lead_time',
                                                                          'arrival_date_year',
                                                                          'arrival_date_week_number',
                                                                          'arrival_date_day_of_month',
                                                                          'stays_in_weekend_nights',
                                                                          'stays_in_week_nights',
                                                                          'adults',
                                                                          'children',
                                                                          'babies',
                                                                          'is_repeated...
                                                                          'arrival_date_month',
                                                                          'meal',
                                                                          'market_segment',
                                                                          'distribution_channel',
                                                                          'reserved_room_type',
                                                                          'deposit_type',
                                                                          'agent',
                                                                          'company',
                                                                          'season',
                                                                          'arrival_day_of_week'])])),
                                       ('model',
                                        DecisionTreeClassifier(random_state=42))]),
             n_jobs=-1,
             param_grid={'model__max_depth': [3, 5, 7, 10, 15, None],
                         'model__min_samples_leaf': [1, 5, 10, 25, 50]},
             return_train_score=True, scoring='roc_auc')

In [24]:
print("Best parameters:")
print(grid_search_dt.best_params_)

print(
    "\nBest mean CV ROC-AUC:",
    round(grid_search_dt.best_score_, 4)
)

Best parameters:
{'model__max_depth': 10, 'model__min_samples_leaf': 10}

Best mean CV ROC-AUC: 0.7712


In [25]:
dt_tuning_results = pd.DataFrame(
    grid_search_dt.cv_results_
)

dt_tuning_table = dt_tuning_results[
    [
        "param_model__max_depth",
        "param_model__min_samples_leaf",
        "mean_test_score",
        "std_test_score",
        "mean_train_score",
        "rank_test_score"
    ]
].copy()

dt_tuning_table = dt_tuning_table.rename(
    columns={
        "param_model__max_depth": "Max Depth",
        "param_model__min_samples_leaf": "Min Samples Leaf",
        "mean_test_score": "Mean CV ROC-AUC",
        "std_test_score": "CV ROC-AUC Std",
        "mean_train_score": "Mean Train ROC-AUC",
        "rank_test_score": "Rank"
    }
)

dt_tuning_table = dt_tuning_table.sort_values(
    "Rank"
).reset_index(drop=True)

dt_tuning_table.head(15)

,Max Depth,Min Samples Leaf,Mean CV ROC-AUC,CV ROC-AUC Std,Mean Train ROC-AUC,Rank
0,10,10,0.771235,0.004911,0.794767,1
1,15,50,0.771012,0.002286,0.811982,2
2,10,5,0.769941,0.005174,0.796433,3
3,10,1,0.769785,0.005372,0.797812,4
4,10,25,0.769674,0.003792,0.791191,5
5,10,50,0.769438,0.003841,0.787334,6
6,15,25,0.767829,0.003804,0.824653,7
7,None,50,0.766715,0.003609,0.826280,8
8,15,10,0.760922,0.004580,0.841001,9
9,7,5,0.760581,0.003510,0.768628,10


## 6. Hyperparameter Tuning Result

The untuned Decision Tree showed severe overfitting. Its mean training ROC-AUC was approximately **0.9997**, while mean cross-validation ROC-AUC was only **0.6541**.

Hyperparameter tuning selected:

- `max_depth = 10`
- `min_samples_leaf = 10`

The tuned configuration achieved a mean cross-validation ROC-AUC of approximately **0.7712** and a mean training ROC-AUC of approximately **0.7948**.

Limiting tree depth and requiring a minimum number of observations in each leaf substantially reduced overfitting while improving cross-validated predictive performance.

The difference between training and cross-validation ROC-AUC decreased from approximately **0.3456** for the untuned tree to approximately **0.0235** for the tuned tree.

In [26]:
best_tree_pipeline = grid_search_dt.best_estimator_

final_dt_cv = cross_validate(
    best_tree_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring={
        "roc_auc": "roc_auc",
        "pr_auc": "average_precision"
    },
    n_jobs=-1,
    return_train_score=True
)

final_dt_table = pd.DataFrame({
    "Fold": range(1, 6),
    "Train ROC-AUC": final_dt_cv["train_roc_auc"],
    "CV ROC-AUC": final_dt_cv["test_roc_auc"],
    "CV PR-AUC": final_dt_cv["test_pr_auc"]
})

final_dt_table

,Fold,Train ROC-AUC,CV ROC-AUC,CV PR-AUC
0,1,0.797336,0.765338,0.539362
1,2,0.795196,0.769322,0.542001
2,3,0.794483,0.770941,0.536796
3,4,0.793717,0.770326,0.542965
4,5,0.793103,0.780248,0.553329


In [27]:
final_dt_roc = final_dt_cv["test_roc_auc"].mean()
final_dt_roc_std = final_dt_cv["test_roc_auc"].std()

final_dt_pr = final_dt_cv["test_pr_auc"].mean()
final_dt_pr_std = final_dt_cv["test_pr_auc"].std()

final_dt_train_roc = final_dt_cv["train_roc_auc"].mean()

print("Final Train ROC-AUC:", round(final_dt_train_roc, 4))
print("Final CV ROC-AUC:", round(final_dt_roc, 4))
print("Final CV ROC-AUC std:", round(final_dt_roc_std, 4))
print("Final CV PR-AUC:", round(final_dt_pr, 4))
print("Final CV PR-AUC std:", round(final_dt_pr_std, 4))

Final Train ROC-AUC: 0.7948
Final CV ROC-AUC: 0.7712
Final CV ROC-AUC std: 0.0049
Final CV PR-AUC: 0.5429
Final CV PR-AUC std: 0.0056


In [28]:
tree_comparison = pd.DataFrame({
    "Model": [
        "Untuned Decision Tree",
        "Tuned Decision Tree"
    ],
    "Train ROC-AUC": [
        baseline_dt_train_roc,
        final_dt_train_roc
    ],
    "CV ROC-AUC": [
        baseline_dt_roc,
        final_dt_roc
    ],
    "CV PR-AUC": [
        baseline_dt_pr,
        final_dt_pr
    ]
})

tree_comparison["Train-CV Gap"] = (
    tree_comparison["Train ROC-AUC"]
    - tree_comparison["CV ROC-AUC"]
)

tree_comparison

,Model,Train ROC-AUC,CV ROC-AUC,CV PR-AUC,Train-CV Gap
0,Untuned Decision Tree,0.999724,0.654100,0.36824,0.345624
1,Tuned Decision Tree,0.794767,0.771235,0.54289,0.023532


In [29]:
fitted_preprocessor = best_tree_pipeline.named_steps["preprocessor"]

feature_names = fitted_preprocessor.get_feature_names_out()

print("Number of transformed features:", len(feature_names))
print(feature_names[:30])

Number of transformed features: 85
['numeric__lead_time' 'numeric__arrival_date_year'
 'numeric__arrival_date_week_number' 'numeric__arrival_date_day_of_month'
 'numeric__stays_in_weekend_nights' 'numeric__stays_in_week_nights'
 'numeric__adults' 'numeric__children' 'numeric__babies'
 'numeric__is_repeated_guest' 'numeric__previous_cancellations'
 'numeric__previous_bookings_not_canceled' 'numeric__adr'
 'numeric__total_nights' 'numeric__total_guests' 'numeric__has_kids'
 'categorical__hotel_City Hotel' 'categorical__hotel_Resort Hotel'
 'categorical__arrival_date_month_April'
 'categorical__arrival_date_month_August'
 'categorical__arrival_date_month_December'
 'categorical__arrival_date_month_February'
 'categorical__arrival_date_month_January'
 'categorical__arrival_date_month_July'
 'categorical__arrival_date_month_June'
 'categorical__arrival_date_month_March'
 'categorical__arrival_date_month_May'
 'categorical__arrival_date_month_November'
 'categorical__arrival_date_month_Octob

In [30]:
fitted_tree = best_tree_pipeline.named_steps["model"]

tree_rules = export_text(
    fitted_tree,
    feature_names=list(feature_names),
    max_depth=5
)

print(tree_rules)

|--- numeric__lead_time <= 11.50
|   |--- categorical__agent_9.0 <= 0.50
|   |   |--- numeric__previous_cancellations <= 0.50
|   |   |   |--- numeric__previous_bookings_not_canceled <= 0.50
|   |   |   |   |--- categorical__deposit_type_Non Refund <= 0.50
|   |   |   |   |   |--- numeric__lead_time <= 1.50
|   |   |   |   |   |   |--- truncated branch of depth 5
|   |   |   |   |   |--- numeric__lead_time >  1.50
|   |   |   |   |   |   |--- truncated branch of depth 5
|   |   |   |   |--- categorical__deposit_type_Non Refund >  0.50
|   |   |   |   |   |--- numeric__arrival_date_day_of_month <= 16.50
|   |   |   |   |   |   |--- class: 0
|   |   |   |   |   |--- numeric__arrival_date_day_of_month >  16.50
|   |   |   |   |   |   |--- class: 1
|   |   |   |--- numeric__previous_bookings_not_canceled >  0.50
|   |   |   |   |--- categorical__hotel_Resort Hotel <= 0.50
|   |   |   |   |   |--- numeric__arrival_date_week_number <= 7.50
|   |   |   |   |   |   |--- class: 0
|   |   |   | 

In [31]:
full_tree_rules = export_text(
    fitted_tree,
    feature_names=list(feature_names)
)

RULES_PATH = MODEL_DIR / "decision_tree_rules.txt"

with open(RULES_PATH, "w") as f:
    f.write(full_tree_rules)

print("Readable rules saved to:")
print(RULES_PATH)

Readable rules saved to:
/content/hotel-cancellation/models/decision_tree_rules.txt


## 7. Why Decision Tree Suits the Problem

A Decision Tree is appropriate for hotel cancellation prediction because it can capture nonlinear relationships and interactions between booking characteristics without requiring assumptions about linear relationships.

It also produces interpretable decision rules, allowing the hotel to understand combinations of booking characteristics associated with higher cancellation risk. Tree complexity must be controlled carefully because unrestricted trees can strongly overfit the training data.

In [32]:
decision_tree_results = pd.DataFrame({
    "Model": ["Decision Tree"],
    "Configuration": [
        "max_depth=10, min_samples_leaf=10"
    ],
    "CV ROC-AUC": [
        final_dt_roc
    ],
    "CV ROC-AUC Std": [
        final_dt_roc_std
    ],
    "CV PR-AUC": [
        final_dt_pr
    ]
})

decision_tree_results

,Model,Configuration,CV ROC-AUC,CV ROC-AUC Std,CV PR-AUC
0,Decision Tree,"max_depth=10, min_samples_leaf=10",0.771235,0.004911,0.54289


In [33]:
MODEL_PATH = MODEL_DIR / "decision_tree_classifier.joblib"

joblib.dump(
    best_tree_pipeline,
    MODEL_PATH
)

print("Model saved to:")
print(MODEL_PATH)

Model saved to:
/content/hotel-cancellation/models/decision_tree_classifier.joblib


In [34]:
loaded_tree = joblib.load(MODEL_PATH)

print("Model loaded successfully.")
print(type(loaded_tree))

Model loaded successfully.
<class 'sklearn.pipeline.Pipeline'>


## 8. Conclusion

The Decision Tree classifier was successfully developed using the shared training data and 5-fold stratified cross-validation.

The unrestricted Decision Tree strongly overfit the training data, achieving a mean training ROC-AUC of 0.9997 but only 0.6541 mean cross-validation ROC-AUC. This produced a large train-to-CV performance gap of approximately 0.3456.

Hyperparameter tuning over `max_depth` and `min_samples_leaf` selected:

- `max_depth = 10`
- `min_samples_leaf = 10`

The tuned Decision Tree achieved a mean CV ROC-AUC of **0.7712 ± 0.0049** and a mean CV PR-AUC of **0.5429 ± 0.0056**.

The mean training ROC-AUC decreased to 0.7948, reducing the train-to-CV gap to approximately 0.0235. This indicates that controlling tree complexity substantially reduced overfitting while improving generalization performance.

The final model uses one-hot encoding for categorical variables and passes numerical variables through without scaling. The fitted pipeline was saved using joblib, and readable decision rules were exported separately for interpretation.

The test dataset was not accessed during model development or tuning.